In [ ]:
import torch
from torch.nn.functional import logsigmoid
import numpy as np
from scipy.stats import norm
from tqdm.autonotebook import tqdm
from IPython.display import clear_output
import os, csv

# set a device + specify number of seeds, uniform-mixing & CI target coverage + spread of budgets
device = "cuda" if torch.cuda.is_available() else "cpu"
N_SEEDS, ALPHA = 100, 0.05
BUDGET_PROPS = np.round(np.linspace(0.0, 0.25, 11)[1:], decimals=3)

# which dataset are we running?
dataset = "mmlu-pro" # or "bbh+gpqa+ifeval+math+musr"

# filename for this simulation
fname = f"{dataset}_baselines.csv"

# dataframe to store all of our results
columns = ["dataset", "budget", "tau", "method", "model_idx", 
           "seed", "thetahat", "coverage", "width", "n_labels"]
if fname not in os.listdir("../logs"):
    with open(f"../logs/{fname}", "wt") as file:
        file.write(",".join(columns) + "\n")
        
# dummy tensors for brevity
ZERO = torch.tensor(0.0, device=device)

In [ ]:
#### LOAD IN OUR DATA

# load in initial factor model fits for LLM evaluations
U = torch.load(f"../datasets/llm_evals/{dataset}/U.pt", map_location=device).to(device) # (N_OLD, D)
V = torch.load(f"../datasets/llm_evals/{dataset}/V.pt", map_location=device).to(device) # (N_Q, D)
ys = torch.load(f"../datasets/llm_evals/{dataset}/Y_test.pt", map_location=device).to(device) # (N_NEW, N_Q)

# get our finite-population size + no. of new models
N_NEW, N_Q = ys.shape; D = U.shape[1]

In [ ]:
#### SEQUENTIAL ACTIVE INFERENCE-TYPE SAMPLING (UNIFORM + PPI)

# iterate through our budgets
for budget_idx, budget in enumerate(BUDGET_PROPS):
    
    # how many labels can we collect?
    n = int(budget * N_Q)
    
    # for LLM evaluations, we are parallelizing across models. So need for-loop over seeds
    for seed in range(N_SEEDS):
        
        # set a seed for reproducibility
        torch.random.manual_seed(858 + seed)
        
        #### initialize data structures
        
        # how many labels have we used up already
        obs_counts = torch.zeros(size=(N_NEW, 1), dtype=torch.float32, device=device) # (N_NEW, 1)

        # recurring sum (WITHOUT NORMALIZATION) of thetahat (need div by N) and sigmahat^2 (need div by N^2)
        thetahats_unnormalized_unif = torch.zeros(
            size=(N_NEW, 1), device=device, dtype=torch.float32) # (N_NEW, 1)
        sigmahats_sq_unnormalized_unif = torch.zeros(
            size=(N_NEW, 1), device=device, dtype=torch.float32) # (N_NEW, 1)
        thetahats_unnormalized_ppi = torch.zeros(
            size=(N_NEW, 1), device=device, dtype=torch.float32) # (N_NEW, 1)
        sigmahats_sq_unnormalized_ppi = torch.zeros(
            size=(N_NEW, 1), device=device, dtype=torch.float32) # (N_NEW, 1)
        
        # initialize new model factors
        Uhats = torch.tile(U.mean(axis=0, keepdim=True), (N_NEW, 1)) # (N_NEW, D)
        diag_precisions = torch.tile(1.0 / (U.std(axis=0, keepdim=True) ** 2), (N_NEW, 1)) # (N_NEW, D)

        #### adaptive sampling rounds

        # loop thru our whole question bank
        for t in tqdm(range(N_Q), desc=f"Budget {budget_idx + 1} of {len(BUDGET_PROPS)}, Seed {seed} of {N_SEEDS}"):

            #### PART 1. computing q_t(j) probabilities + sampling probabilities

            # under uniform/ppi sampling, pi_t = n / N always.
            pi_t = n / N_Q

            # f. are we going to query this datapoint or not?
            xi_t = torch.bernoulli(
                torch.full(size=(N_NEW, 1), fill_value=pi_t, device=device)) # (N_NEW, 1)
            
            #### PART 2. UPDATING SEQUENTIAL ACTIVE INFERENCE-TYPE ESTIMATOR + BOOKKEEPING
            
            # a. compute the uncertainty scores based on the current ML prediction
            logits = Uhats @ V.T # (N_NEW, N_Q)
            fxs = torch.sigmoid(logits) # (N_NEW, N_Q)

            # b. update our means and variances
            thetahats_unnormalized_unif += ((xi_t / pi_t) * ys[:,t:t+1])
            sigmahats_sq_unnormalized_unif += (xi_t * (1.0 - pi_t) / (pi_t ** 2) ) * (ys[:,t:t+1] ** 2)
            thetahats_unnormalized_ppi += ((xi_t / pi_t) * (ys[:,t:t+1] - fxs[:,t:t+1])) + fxs[:,t:t+1]
            sigmahats_sq_unnormalized_ppi += (xi_t * (1.0 - pi_t) / (pi_t ** 2) ) * ((ys[:,t:t+1] - fxs[:,t:t+1]) ** 2)

            # c. did we exhaust budget? also, did we hit a big-residual datapoint?
            obs_counts += xi_t

            #### PART 3: updating our online logistic regression
            
            # a. convert xi_t to bool
            xi_t = xi_t.to(bool).squeeze() # (N_NEW,)

            # b. fhat * (1 - fhat); note that diag(xx^T) = (x_1^2, \dots, x_D^2)
            w_Is = fxs[:,t:t+1][xi_t] * (1.0 - fxs[:,t:t+1][xi_t]) # (N_FLIP_HEADS, 1)
            x_t = V[t] # (D,)
            diag_precisions[xi_t] += w_Is * (x_t ** 2) # (N_FLIP_HEADS, D)
            Uhats[xi_t] += (ys[:,t:t+1][xi_t] - fxs[:,t:t+1][xi_t]) * x_t / diag_precisions[xi_t] # (N_FLIP_HEADS, D)

        #### COMPUTING UNIF + PPI ESTIMATOR SUMMARIES
        
        #### 1. UNIFORM ESTIMATOR

        # a. get our mean estimate + variance estimate
        thetahats_unif = thetahats_unnormalized_unif / N_Q
        sigmahats_unif = torch.sqrt(torch.maximum(sigmahats_sq_unnormalized_unif, ZERO) / N_Q)
        sigmahats_bar_unif = sigmahats_unif / torch.sqrt(torch.tensor(N_Q, device=device))

        # b. get our z-score + the lower/upper limits of our interval
        z_score = norm.ppf(1 - (ALPHA / 2))
        lbs = thetahats_unif - z_score * sigmahats_bar_unif
        ubs = thetahats_unif + z_score * sigmahats_bar_unif
        widths = (ubs - lbs).cpu().numpy()
        coverages = ((lbs <= ys.mean(dim=-1, keepdim=True)) & (ys.mean(dim=-1, keepdim=True) <= ubs)).cpu().numpy()

        # c. for file writing purposes, convert thetahats + obs_counts to numpy
        thetahats_unif = thetahats_unif.cpu().numpy()
        obs_counts = obs_counts.cpu().numpy()

        # format results + directly write to file
        results = [
            [dataset, budget, 0.0, "uniform", model_idx, seed, 
             thetahats_unif[model_idx][0], 
             coverages[model_idx][0], widths[model_idx][0], obs_counts[model_idx][0]
            ] for model_idx in range(N_NEW)]
        with open(f"../logs/{fname}", "a", newline="") as file:
            writer = csv.writer(file)
            writer.writerows(results)
            
        #### 2. PPI ESTIMATOR

        # a. get our mean estimate + variance estimate
        thetahats_ppi = thetahats_unnormalized_ppi / N_Q
        sigmahats_ppi = torch.sqrt(torch.maximum(sigmahats_sq_unnormalized_ppi, ZERO) / N_Q)
        sigmahats_bar_ppi = sigmahats_ppi / torch.sqrt(torch.tensor(N_Q, device=device))

        # b. get our z-score + the lower/upper limits of our interval
        z_score = norm.ppf(1 - (ALPHA / 2))
        lbs = thetahats_ppi - z_score * sigmahats_bar_ppi
        ubs = thetahats_ppi + z_score * sigmahats_bar_ppi
        widths = (ubs - lbs).cpu().numpy()
        coverages = ((lbs <= ys.mean(dim=-1, keepdim=True)) & (ys.mean(dim=-1, keepdim=True) <= ubs)).cpu().numpy()

        # c. for file writing purposes, convert thetahats ppi also to numpy
        thetahats_ppi = thetahats_ppi.cpu().numpy()

        # format results + directly write to file
        results = [
            [dataset, budget, 0.0, "ppi", model_idx, seed, 
             thetahats_ppi[model_idx][0], 
             coverages[model_idx][0], widths[model_idx][0], obs_counts[model_idx][0]
            ] for model_idx in range(N_NEW)]
        with open(f"../logs/{fname}", "a", newline="") as file:
            writer = csv.writer(file)
            writer.writerows(results)
            
        # clean house
        clear_output(wait=True)